In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gmbiggan/agroai-iot-synthetic-testing-data-fake-not-real/agroai_synthetic_data.csv


In [2]:
# =============================================
# AGROAI XGBOOST MODEL TRAINING
# =============================================
# 📌 Dataset: Synthetic IoT data (Generated, not real)
# 🎯 Goal: Predict Soil Moisture (Next 15 min)
# 👨‍💻 Author: G.M. Biggan - AgroAI Project

import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD DATA (Kaggle Dataset)

In [6]:
# আপনি আপনার ডেটাসেটের পাথ দেবেন
df = pd.read_csv('/kaggle/input/datasets/gmbiggan/agroai-iot-synthetic-testing-data-fake-not-real/agroai_synthetic_data.csv')

In [7]:
print("="*50)
print("AGROAI DATASET INFO")
print("="*50)
print(f"📊 Total Records: {len(df)}")
print(f"📅 Date Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"🔧 Columns: {df.columns.tolist()}")
print("\n📋 First 5 rows:")
print(df.head())

AGROAI DATASET INFO
📊 Total Records: 672
📅 Date Range: 2024-01-01 00:00:00 to 2024-01-07 23:45:00
🔧 Columns: ['timestamp', 'temperature', 'humidity', 'soil_moisture', 'water_level', 'light_intensity', 'soil_ph', 'pump_status', 'battery']

📋 First 5 rows:
             timestamp  temperature   humidity  soil_moisture  water_level  \
0  2024-01-01 00:00:00    16.225165  80.249314      50.092896   350.000000   
1  2024-01-01 00:15:00    15.000000  72.874918      50.111638   349.153894   
2  2024-01-01 00:30:00    15.000000  72.480420      49.948941   347.880900   
3  2024-01-01 00:45:00    15.000000  77.466616      50.686855   346.951294   
4  2024-01-01 01:00:00    15.000000  62.608704      50.254894   346.229454   

   light_intensity   soil_ph  pump_status    battery  
0         4.356331  6.144297            0  98.000000  
1         4.064398  6.779970            0  97.713032  
2         1.109375  6.507574            0  97.570344  
3         1.340644  6.294234            0  97.427044  
4

2. FEATURE ENGINEERING

In [9]:
# টাইমস্ট্যাম্প থেকে ফিচার তৈরি
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

In [10]:
# ল্যাগ ফিচার (পূর্বের মান) - Time series এর জন্য গুরুত্বপূর্ণ
df['soil_moisture_lag1'] = df['soil_moisture'].shift(1)
df['soil_moisture_lag2'] = df['soil_moisture'].shift(2)
df['temperature_lag1'] = df['temperature'].shift(1)
df['humidity_lag1'] = df['humidity'].shift(1)

In [11]:
# NaN ভ্যালু মুছে ফেলা (ল্যাগ ফিচারের জন্য)
df = df.dropna().reset_index(drop=True)

print(f"\n🔄 After feature engineering: {len(df)} records")


🔄 After feature engineering: 670 records


 3. FEATURES & TARGET

In [14]:

# ------------------------------
# টার্গেট: পরবর্তী সময়ের মাটির আর্দ্রতা
feature_columns = [
    'temperature',
    'humidity', 
    'soil_moisture',
    'light_intensity',
    'soil_ph',
    'hour',
    'day_of_week',
    'is_weekend',
    'soil_moisture_lag1',
    'soil_moisture_lag2',
    'temperature_lag1',
    'humidity_lag1'
]

X = df[feature_columns].values
y = df['soil_moisture'].values  # predicted soil moisture

print(f"\n🎯 Features: {len(feature_columns)} columns")
print(f"📈 Target: Soil Moisture")


🎯 Features: 12 columns
📈 Target: Soil Moisture


4. TRAIN-TEST SPLIT

In [15]:
# 4. TRAIN-TEST SPLIT
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False  # Time series ক্রম বজায় রাখা
)

print(f"\n🔄 Train: {len(X_train)} records")
print(f"🧪 Test: {len(X_test)} records")



🔄 Train: 536 records
🧪 Test: 134 records


In [17]:
# 5. SCALING
# ------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [19]:
# 6. XGBOOST MODEL TRAINING
# ------------------------------
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmse'
)

print("\n🚀 Training XGBoost Model...")
model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=False
)


🚀 Training XGBoost Model...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [20]:
# 7. MODEL EVALUATION
# ------------------------------
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

train_mae = mean_absolute_error(y_train, y_pred_train)
test_mae = mean_absolute_error(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("\n" + "="*50)
print("📊 MODEL PERFORMANCE")
print("="*50)
print(f"📌 Training MAE: {train_mae:.4f}")
print(f"📌 Testing MAE: {test_mae:.4f}")
print(f"📌 Training R² Score: {train_r2:.4f}")
print(f"📌 Testing R² Score: {test_r2:.4f}")
print("="*50)


📊 MODEL PERFORMANCE
📌 Training MAE: 0.0276
📌 Testing MAE: 0.2975
📌 Training R² Score: 1.0000
📌 Testing R² Score: 0.9916


In [21]:
# 8. FEATURE IMPORTANCE
# ------------------------------
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🎯 TOP 5 IMPORTANT FEATURES:")
print(feature_importance.head(5).to_string(index=False))


🎯 TOP 5 IMPORTANT FEATURES:
           feature  importance
     soil_moisture    0.839967
              hour    0.058780
soil_moisture_lag2    0.022526
     humidity_lag1    0.018982
soil_moisture_lag1    0.012636


In [22]:
# 9. SAVE MODEL & SCALER
# ------------------------------
joblib.dump(model, 'agroai_xgboost_model.pkl')
joblib.dump(scaler, 'agroai_scaler.pkl')

print("\n💾 Model saved: agroai_xgboost_model.pkl")
print("💾 Scaler saved: agroai_scaler.pkl")



💾 Model saved: agroai_xgboost_model.pkl
💾 Scaler saved: agroai_scaler.pkl


In [24]:
# 10. TEST PREDICTION FUNCTION
# ------------------------------
def predict_soil_moisture(temperature, humidity, current_soil, light_intensity, soil_ph,
                          soil_lag1, soil_lag2, temp_lag1, hum_lag1, hour, day, is_weekend):
    """ভবিষ্যৎ মাটির আর্দ্রতা প্রেডিক্ট করুন"""
    features = np.array([[
        temperature, humidity, current_soil, light_intensity, soil_ph,
        hour, day, is_weekend,
        soil_lag1, soil_lag2, temp_lag1, hum_lag1
    ]])
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)[0]
    
    # পাম্প রিকমেন্ডেশন
    pump_recommendation = "ON" if prediction < 35 else "OFF"
    
    return {
        'predicted_soil_moisture': round(prediction, 2),
        'pump_recommendation': pump_recommendation,
        'confidence': 0.85
    }

# টেস্ট কল
print("\n🧪 Test Prediction:")
test_result = predict_soil_moisture(
    temperature=28.5, humidity=65.0, current_soil=45.0,
    light_intensity=75.0, soil_ph=6.8,
    soil_lag1=46.0, soil_lag2=47.0, temp_lag1=28.0, hum_lag1=64.0,
    hour=14, day=2, is_weekend=0
)
print(f"📈 Result: {test_result}")

print("\n✅ Model training complete! Download the .pkl files from Output section.")


🧪 Test Prediction:
📈 Result: {'predicted_soil_moisture': np.float32(44.73), 'pump_recommendation': 'OFF', 'confidence': 0.85}

✅ Model training complete! Download the .pkl files from Output section.
